# Deep Past Challenge — Akkadian-to-English Translation

**TWO-MODEL ENSEMBLE | Hybrid MBR (Merged Pools, Uniform Average) + Epsilon Sampling**

This pipeline implements a highly optimized two-model ensemble utilizing a **Hybrid Minimum Bayes Risk (MBR)** decoding strategy. It completely removes explicit cross-model bonuses in favor of implicit frequency signals, while reintroducing epsilon sampling to maximize translation diversity and coverage.

## Score History
* **v13 (34.3):** Single-model, composite MBR
* **v16 (34.8):** Two-model, pure chrF++ MBR
* **v17 (35.4):** Two-model, geo-mean(BLEU × chrF++) MBR
* **v19 (35.5):** Pipeline + standard agreement bonus
* **v20 (35.6):** Geo-mean, separate pools, cross-model bonus, ε×6
* **v39 (35.4):** Composite, separate pools, cross-model bonus, ε×6
* **v40 (35.6):** Composite, separate pools, cross-model bonus, temps [0.55,0.75,0.95], ε×6
* **v41 (36.2):** Composite, MERGED pools, NO cross-model bonus, temps [0.55,0.75,0.95], ε×6

## What v41 Adds: Implicit Consensus & Epsilon Diversity

Previous implementations (like v20) applied a hardcoded `+0.12` bonus when both models produced the exact same string. However, testing revealed that when combined with a composite pairwise metric, this explicit bonus was calibrated to the wrong utility scale and actually hurt performance by double-counting the agreement signal. 

v41 fixes this by adopting the "Hybrid" approach, with a critical addition:
* **The Implicit Signal (Merged Pools):** Pools are merged BEFORE postprocessing and deduplication. If both models independently produce the same translation, it appears multiple times in the combined raw list. When evaluated via a uniform pairwise average, these cross-model candidates naturally score higher because more references in the pool resemble them.
* **The Epsilon Advantage:** The 35.7 Hybrid baseline removed epsilon sampling. v41 proves that epsilon sampling ($\varepsilon=0.02$) adds vital diversity that increases the pool's coverage of correct translations. Because v41 uses a simple uniform average (rather than count-weighting), these low-frequency epsilon outputs are no longer unfairly penalized during utility computation.

## Scoring Formula

The explicitly calculated `cross_model_agreed` and `within_model` counts are entirely absent. The selector calculates an exact, simple uniform average over all $n-1$ unique references in the deduplicated pool:

$$Utility(h_i, h_j) = \frac{0.55 \cdot \text{chrF++}(h_i, h_j) + 0.25 \cdot \text{BLEU}(h_i, h_j) + 0.20 \cdot \text{Jaccard}(h_i, h_j)}{1.0}$$

$$FinalScore(h_i) = \left( \frac{1}{N-1} \sum_{j \neq i} Utility(h_i, h_j) \right) + 0.10 \cdot GaussianLengthBonus(h_i)$$

*(Note: The Gaussian length bonus is an additive bell curve centered on the median pool length, pushing scores by up to 10 points to break ties without overriding the pairwise quality.)*

## Generation Changes Over v40 & Hybrid
To create a perfectly balanced pool that captures both the best statistical guesses and highly diverse edge-cases, the candidate generation precisely mimics the Hybrid's volume but replaces some generic sampling with targeted epsilon sampling:
* **Beam Search:** 4 candidates.
* **Nucleus Sampling:** Wider temperature spread `[0.55, 0.75, 0.95]` $\times 2 = 6$ candidates.
* **Epsilon Samples:** Added back 6 candidates ($\varepsilon=0.02$).

**Total Pool Composition:** 4 beam + 6 nucleus + 6 $\varepsilon$ = 16 candidates per model $\times$ 2 models = **~32 total candidates per sentence** (capped exactly at `pool_cap=32`).

In [1]:
#!/usr/bin/env python3
from __future__ import annotations

import gc, json, logging, math, os, random, re, subprocess, importlib, sys
from contextlib import nullcontext
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset, Sampler
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

import warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR   = Path("/kaggle/working")

In [2]:
# ------------------
#   Path discovery
# ------------------

# Recursively traverse the input root to print the directory tree and formatted file sizes.
print("=" * 65)
for root, dirs, files in os.walk(INPUT_ROOT):
    depth = root.replace(str(INPUT_ROOT), "").count(os.sep)
    print(f"{'  '*depth}{Path(root).name}/")
    for fn in sorted(files):
        sz = os.path.getsize(os.path.join(root, fn))
        print(f"{'  '*(depth+1)}{fn}  "
              f"({'%.1f'%(sz/1e9)+' GB' if sz>1e8 else str(sz//1024)+' KB'})")
print("=" * 65)

# Traverse the file system under root to resolve the exact path of a target file.
def _find_file(root, *names):
    for dp, _, fns in os.walk(root):
        for n in names:
            if n in fns: return Path(dp) / n
    return None

# Collect a unique list of folder paths that hold offline pip installation files.
def _find_whl_dirs(root):
    s = set()
    for dp, _, fns in os.walk(root):
        if any(f.endswith(".whl") for f in fns): s.add(Path(dp))
    return list(s)

# Discover valid model checkpoints and order them so the primary optimized model becomes MODEL_A.
def _find_model_dirs(root) -> List[Path]:
    MARKERS = {"config.json", "pytorch_model.bin", "model.safetensors"}
    found: List[Path] = []
    for dp, _, files in os.walk(root):
        if MARKERS & set(files):
            p = Path(dp)
            if p not in found: found.append(p)
    def _sf_size(p: Path) -> int:
        sf = p / "model.safetensors"
        return sf.stat().st_size if sf.exists() else 0
    found.sort(key=_sf_size, reverse=True)
    if (len(found) >= 2
            and "mbr" in str(found[0]).lower()
            and "optimized" not in str(found[0]).lower()):
        found[0], found[1] = found[1], found[0]
    return found

# Scan the input directory to find the specific files and folders required for inference.
TEST_FILE  = _find_file(INPUT_ROOT, "test.csv", "test_sentences.csv")
WHL_DIRS   = _find_whl_dirs(INPUT_ROOT)
ALL_MODELS = _find_model_dirs(INPUT_ROOT)

# Ensure critical assets were found before proceeding.
if not TEST_FILE:  raise FileNotFoundError("test.csv not found.")
if not ALL_MODELS: raise FileNotFoundError("No model checkpoint found.")

# Assign models to their primary/ensemble roles and set the execution mode.
MODEL_A        = str(ALL_MODELS[0])
MODEL_B        = str(ALL_MODELS[1]) if len(ALL_MODELS) > 1 else None
TWO_MODEL_MODE = MODEL_B is not None

# Log the discovered models, their sizes, and warn if falling back to single-model mode.
print("\n" + "═"*65)
print("  MODEL DISCOVERY")
print("═"*65)
for i, mp in enumerate(ALL_MODELS):
    sf = mp / "model.safetensors"
    gb = sf.stat().st_size / 1e9 if sf.exists() else 0
    print(f"  {'MODEL_A (base)' if i==0 else 'MODEL_B (ensemble)'}: "
          f"{mp.relative_to(INPUT_ROOT)}  ({gb:.2f} GB)")
if not TWO_MODEL_MODE:
    print("  ⚠️  WARNING: Single-model mode — attach a second model.")
print("═"*65)

input/
  competitions/
    deep-past-initiative-machine-translation/
      OA_Lexicon_eBL.csv  (3470 KB)
      Sentences_Oare_FirstWord_LinNum.csv  (1939 KB)
      bibliography.csv  (148 KB)
      eBL_Dictionary.csv  (1582 KB)
      publications.csv  (0.6 GB)
      published_texts.csv  (10878 KB)
      resources.csv  (94 KB)
      sample_submission.csv  (0 KB)
      test.csv  (0 KB)
      train.csv  (1589 KB)
  datasets/
    francescocampigotto/
      offline-pkgs/
        offline_pkgs/
          peft-0.18.1-py3-none-any.whl  (543 KB)
          portalocker-3.2.0-py3-none-any.whl  (21 KB)
          sacrebleu-2.6.0-py3-none-any.whl  (98 KB)
    spencevanasperdt/
      mattiaangelibyt5-akkadian-mbrpytorchdefault6/
        added_tokens.json  (2 KB)
        config.json  (0 KB)
        generation_config.json  (0 KB)
        model.safetensors  (2.3 GB)
        special_tokens_map.json  (3 KB)
        tokenizer_config.json  (25 KB)
    assiaben/
      final-byt5/
        byt5-akkadian-optimized

In [3]:
# --------------------
#   Offline Packages
# --------------------

# Check if a package is already installed.
def _pkg_ok(n):
    try: importlib.import_module(n); return True
    except ImportError: return False

# Attempt an offline pip installation from local wheel files.
def _try_install(pkg):
    for d in WHL_DIRS:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "--no-index", f"--find-links={d}", pkg],
                           capture_output=True, text=True)
        if r.returncode == 0: importlib.invalidate_caches(); return True
    return False

# Install specific required packages if they are missing.
for pkg in ("portalocker", "sacrebleu"):
    if not _pkg_ok(pkg):
        ok = _try_install(pkg)
        print(f"  {'✅' if ok else '⚠️'} {pkg} {'installed' if ok else 'fallback'}")

# Set a global flag to indicate if sacrebleu is available for metric scoring.
try:
    import sacrebleu as _sacrebleu; SACREBLEU_OK = True; print("✅ sacrebleu ready.")
except ImportError:
    SACREBLEU_OK = False; print("sacrebleu fallback.")

  ✅ portalocker installed
  ✅ sacrebleu installed
✅ sacrebleu ready.


In [4]:
# -------------------------------
#  Pure-Python fallback metrics
# -------------------------------

# Yield an overlapping sliding window of size n to build a Counter dictionary for n-gram matching.
def _ngrams_c(tokens, n):
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))

# Compute the chrF++ metric by evaluating character and word n-gram precision and recall.
def _py_chrf(a, b, char_order=6, word_order=2):
    if not a or not b: return 0.0
    ps, rs = [], []
    for cn in range(1, char_order+1):
        hg = Counter(a[i:i+cn] for i in range(len(a)-cn+1))
        rg = Counter(b[i:i+cn] for i in range(len(b)-cn+1))
        m = sum(min(c, rg[g]) for g, c in hg.items())
        ps.append(m/max(1, sum(hg.values()))); rs.append(m/max(1, sum(rg.values())))
    for wn in range(1, word_order+1):
        hg = _ngrams_c(a.split(), wn); rg = _ngrams_c(b.split(), wn)
        m = sum(min(c, rg[g]) for g, c in hg.items())
        ps.append(m/max(1, sum(hg.values()))); rs.append(m/max(1, sum(rg.values())))
    ap = sum(ps)/len(ps); ar = sum(rs)/len(rs); d = 4*ap + ar
    return 100.0*5*ap*ar/d if d > 0 else 0.0

# Evaluate translation quality using a smoothed BLEU metric.
def _py_sbleu(hyp, ref):
    if not hyp or not ref: return 0.0
    hw, rw = hyp.split(), ref.split()
    if not hw or not rw: return 0.0
    bp = 1.0 if len(hw) >= len(rw) else math.exp(1 - len(rw)/max(1, len(hw)))
    log_avg = 0.0; max_n = min(4, len(hw), len(rw))
    if max_n == 0: return 0.0
    for n in range(1, max_n+1):
        hg = _ngrams_c(hw, n); rg = _ngrams_c(rw, n)
        m = sum(min(c, rg[g]) for g, c in hg.items())
        num = m+1 if n > 1 else m; den = max(1, len(hw)-n+1) + (1 if n > 1 else 0)
        log_avg += math.log(max(num/den, 1e-10))
    return 100.0 * bp * math.exp(log_avg/max_n)

In [5]:
# -----------------
#   Device / BF16 
# -----------------

# Check if the current GPU supports bfloat16 mixed precision.
def _cuda_bf16():
    if not torch.cuda.is_available(): return False
    try: return bool(getattr(torch.cuda, "is_bf16_supported", lambda: False)())
    except: return False

def _bf16_ctx(device, enabled):
    if enabled and device.type == "cuda" and _cuda_bf16():
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()

# Set the global compute device and print its hardware specs.
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BF16 = _cuda_bf16()
print(f"  Device: {DEVICE}"+(f" / {torch.cuda.get_device_name(0)}"
      f" / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB"
      if torch.cuda.is_available() else ""))

# Set random seeds across all libraries for reproducibility.
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed(42)

  Device: cuda / Tesla T4 / 15.6 GB


In [6]:
# ----------
#   Config
# ----------

@dataclass
class Config:
    model_a_path:   str = MODEL_A
    model_b_path:   str = MODEL_B
    test_data_path: str = str(TEST_FILE)
    output_dir:     str = str(WORK_DIR)

    max_input_length: int = 512
    max_new_tokens:   int = 384
    batch_size:       int = 2
    num_workers:      int = 2
    num_buckets:      int = 6

    # v13-proven generation params — DO NOT CHANGE
    num_beam_cands:     int   = 4
    num_beams:          int   = 8
    length_penalty:     float = 1.3
    repetition_penalty: float = 1.2
    early_stopping:     bool  = True
    # no_repeat_ngram_size ABSENT (Akkadian formulaic repetitions must survive)

    # Hybrid's temperature set — wider spread gives more diverse candidates
    use_sampling:        bool        = True
    sample_temperatures: List[float] = field(
        default_factory=lambda: [0.55, 0.75, 0.95])
    num_sample_per_temp: int   = 2
    top_p:               float = 0.92

    # Epsilon sampling — our proven diversity advantage, not used by hybrid
    use_epsilon: bool  = True
    epsilon:     float = 0.02
    num_epsilon: int   = 6

    # v41 MBR: hybrid's exact weights (from the 35.7 notebook)
    # pairwise = (w_chrf×chrF++ + w_bleu×BLEU + w_jaccard×Jaccard) / norm
    mbr_w_chrf:    float = 0.55
    mbr_w_bleu:    float = 0.25
    mbr_w_jaccard: float = 0.20
    # Gaussian length bonus weight (additive, from hybrid 35.7)
    mbr_w_length:  float = 0.10
    # Pool cap — same as hybrid
    mbr_pool_cap:  int   = 32

    use_bf16_amp:           bool = USE_BF16
    use_better_transformer: bool = True
    use_bucket_batching:    bool = True
    use_adaptive_beams:     bool = True
    checkpoint_freq:        int  = 200

    @property
    def num_sample_cands(self): return len(self.sample_temperatures)*self.num_sample_per_temp
    @property
    def num_eps_cands(self): return self.num_epsilon if self.use_epsilon else 0
    @property
    def cands_per_model(self): return self.num_beam_cands+self.num_sample_cands+self.num_eps_cands

    def __post_init__(self):
        Path(self.output_dir).mkdir(exist_ok=True, parents=True)
        assert self.num_beams >= self.num_beam_cands

In [7]:
# -----------
#   Logging
# -----------

def setup_logging(output_dir, version):
    for h in logging.root.handlers[:]: logging.root.removeHandler(h)
    logging.basicConfig(level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        handlers=[logging.StreamHandler(),
                  logging.FileHandler(Path(output_dir)/f"inference_{version}.log")])
    return logging.getLogger(f"v{version}")

# Akkadian Text Preprocessing Pipeline

This preprocessing pipeline cleans and standardizes raw Akkadian transliterations to reduce vocabulary sparsity, handle scholarly transcription variations, and ensure consistent input formatting. Because the ByT5 model operates at the byte level, eliminating inconsistent character representations is critical for model performance.

The pipeline utilizes **vectorized Pandas string operations** for fast execution across batches, performing the following sequential steps:

**1. Diacritic Normalization (`_diacritics`):**
* Converts legacy ASCII digraphs into standard Akkadian Unicode characters (e.g., sz → š, s, → ṣ, t, → ṭ).
* Translates numeric index vowels (both standard digits and subscripts) into their proper accented forms (e.g., a2/a₂ → á, a3/a₃ → à).

**2. Determinative & Logogram Formatting (`_DU`, `_DL`):**
* Sumerograms (uppercase text) enclosed in parentheses are stripped of their parentheses.
* Lowercase text in parentheses (typically determinatives or phonetic complements) is converted to use curly braces {}.

**3. Gap and Broken Text Standardization (`_ngaps`):**
* Uses regular expressions to capture a wide variety of scholarly annotations used for missing, broken, or unreadable text (e.g., ..., [x], (large break), big gap, x x x).
* Collapses all these variations into a single, unified `<gap>` token.

**4. Character Transliteration & Simplification (`_CT`):**
* Maps specific consonants to simplified forms (e.g., ḫ → h, Ḫ → H).
* Removes the aleph (ʾ) and the ₓ symbol.
* Converts subscript numerals into standard digits and standardizes dashes.

**5. Abbreviation Expansion & Number Canonicalization:**
* Expands the common abbreviation KÙ.B. to the full Sumerogram KÙ.BABBAR (silver).
* Converts specific decimal strings into Unicode fraction characters (e.g., 0.5 → ½, 0.8333 → ⅚).
* Uses `_canon_dec` to round and format long floating-point numbers into cleaner fractions or decimals, preserving the semantics of ancient weights and measures.

**6. Whitespace Cleanup:**
* Compresses multiple consecutive spaces into a single space and strips leading and trailing whitespace.

In [8]:
# -----------------
#   Preprocessing
# -----------------

_V2=re.compile(r"([aAeEiIuU])(?:2|₂)"); _V3=re.compile(r"([aAeEiIuU])(?:3|₃)")
_ACUTE=str.maketrans({"a":"á","e":"é","i":"í","u":"ú","A":"Á","E":"É","I":"Í","U":"Ú"})
_GRAVE=str.maketrans({"a":"à","e":"è","i":"ì","u":"ù","A":"À","E":"È","I":"Ì","U":"Ù"})
def _diacritics(s):
    s=s.replace("sz","š").replace("SZ","Š").replace("s,","ṣ").replace("S,","Ṣ")
    s=s.replace("t,","ṭ").replace("T,","Ṭ")
    s=_V2.sub(lambda m:m.group(1).translate(_ACUTE),s)
    return _V3.sub(lambda m:m.group(1).translate(_GRAVE),s)

_FRACS=[(1/6,"0.16666"),(1/4,"0.25"),(1/3,"0.33333"),(1/2,"0.5"),
        (2/3,"0.66666"),(3/4,"0.75"),(5/6,"0.83333")]
_FRAC_TOL=2e-3; _FLOAT_RE=re.compile(r"(?<![\w/])(\d+\.\d{4,})(?![\w/])")
def _canon_dec(x):
    ip=int(math.floor(x+1e-12)); frac=x-ip
    best=min(_FRACS,key=lambda t:abs(frac-t[0]))
    if abs(frac-best[0])<=_FRAC_TOL:
        dec=best[1]; return dec if ip==0 else (f"{ip}{dec[1:]}" if dec.startswith("0.") else f"{ip}+{dec}")
    return f"{x:.5f}".rstrip("0").rstrip(".")

_WS_RE=re.compile(r"\s+")
_GAP_RE=re.compile(
    r"<\s*big[\s_\-]*gap\s*>|<\s*gap\s*>|\bbig[\s_\-]*gap\b|\bx(?:\s+x)+\b"
    r"|\.{3,}|…+|\[\.+\]|\[\s*x\s*\]|\(\s*x\s*\)|(?<!\w)x{2,}(?!\w)|(?<!\w)x(?!\w)"
    r"|\(\s*large\s+break\s*\)|\(\s*break\s*\)|\(\s*\d+\s+broken\s+lines?\s*\)",re.I)
def _ngaps(s): return s.str.replace(_GAP_RE,"<gap>",regex=True)
_CT=str.maketrans({"ḫ":"h","Ḫ":"H","ʾ":"","₀":"0","₁":"1","₂":"2","₃":"3",
    "₄":"4","₅":"5","₆":"6","₇":"7","₈":"8","₉":"9","—":"-","–":"-"})
_U_UP=r"A-ZŠṬṢḪ\u00C0-\u00D6\u00D8-\u00DE\u0160\u1E00-\u1EFF"
_U_LO=r"a-zšṭṣḫ\u00E0-\u00F6\u00F8-\u00FF\u0161\u1E01-\u1EFF"
_DU=re.compile(r"\((["+_U_UP+r"0-9]{1,6})\)"); _DL=re.compile(r"\((["+_U_LO+r"]{1,4})\)")
_PN_RE=re.compile(r"\bPN\b"); _KB_RE=re.compile(r"KÙ\.B\.")
_EF_RE=re.compile(r"0\.8333|0\.6666|0\.3333|0\.1666|0\.625|0\.75|0\.25|0\.5")
_EF_MAP={"0.8333":"⅚","0.6666":"⅔","0.3333":"⅓","0.1666":"⅙","0.625":"⅝","0.75":"¾","0.25":"¼","0.5":"½"}
def _fr(m): return _EF_MAP.get(m.group(0), m.group(0))

class OptimizedPreprocessor:
    def preprocess_batch(self,texts):
        s=pd.Series(texts).fillna("").astype(str).apply(_diacritics)
        s=s.str.replace(_DU,r"\1",regex=True).str.replace(_DL,r"{\1}",regex=True)
        s=_ngaps(s).str.translate(_CT).str.replace("ₓ","",regex=False)
        s=s.str.replace(_KB_RE,"KÙ.BABBAR",regex=True).str.replace(_EF_RE,_fr,regex=True)
        s=s.str.replace(_FLOAT_RE,lambda m:_canon_dec(float(m.group(1))),regex=True)
        return s.str.replace(_WS_RE," ",regex=True).str.strip().tolist()

# Akkadian Text Postprocessing Pipeline

This postprocessing pipeline refines the English translations generated by the ByT5 model. It uses **vectorized Pandas string operations** to efficiently remove residual model artifacts, expand specific domain terminology, standardize ancient weights and dates, and clean up repetitive phrases often produced by beam search.

The pipeline performs the following sequential operations:

**1. Entity and Gap Normalization:**
* Applies the same `<gap>` standardization as the preprocessor to catch any model-generated gap artifacts.
* Masks generic personal name placeholders (`PN`) with `<gap>`.
* Expands hyphenated commodity abbreviations into their full Akkadian-English translations (e.g., `-gold` → `pašallum gold`, `-tax` → `šadduātum tax`, `-textiles` → `kutānum textiles`).

**2. Ancient Weight Conversions (`_SK`):**
* Detects complex, literal translations of fractional shekels and converts them into the correct historical terminology (e.g., `5 11/12 shekels` → `6 shekels less 15 grains`, `5/12 shekel` → `⅓ shekel 15 grains`).

**3. Number Canonicalization:**
* Converts decimal outputs back into Unicode fraction characters (`0.5` → `½`) and rounds long floating-point artifacts using `_canon_dec`.

**4. Annotation and Artifact Cleanup:**
* Removes grammatical annotations the model might have reproduced from training data (e.g., `(fem. pl.)`, `(sing.)`).
* Strips uncertainty markers `(?)` and stray angle brackets (`<< >>`, `< >`), explicitly preserving valid `<gap>` tags.
* Cleans up orphaned punctuation, slashes, and floating letters.

**5. Quote and Date Standardization:**
* Normalizes smart quotes (“ ”, ‘ ’) into standard straight quotes (" ", ' ').
* Converts Roman numeral month formats into standard Arabic numerals (e.g., `Month XII` → `Month 12`).

**6. Character Filtering and Protection:**
* Merges consecutive `<gap>` tokens into a single gap.
* Safely removes forbidden characters (`——<>⌈⌋⌊[]+ʾ;`) while explicitly protecting parentheses `()`, as they are semantically important in the ground truth translations. Uses a temporary hexadecimal placeholder (`\x00G\x00`) to shield `<gap>` tags during this deletion step.
* Transliterates any remaining `ḫ` or `Ḫ` into `h` and `H`.

**7. Beam Search Repetition Removal:**
* Uses advanced regex backreferences to detect and delete repeating words (e.g., "the the" → "the").
* Iterates backwards from 4 to 2 to catch and collapse multi-word repeating phrases (a common artifact in sequence-to-sequence generation).
* Strips spaces before punctuation and collapses repetitive punctuation (e.g., `,,` → `,`).

**8. Final Whitespace Cleanup:**
* Compresses multiple consecutive spaces into a single space and strips leading/trailing whitespace.

In [9]:
# ------------------
#   Postprocessing
# ------------------

_SG_RE=re.compile(r"\(\s*(?:fem|plur|pl|sing|singular|plural|\?|\!)(?:\.\s*(?:plur|plural|sing|singular))?\.?\s*[^)]*\)",re.I)
_BG_RE=re.compile(r"(?<!\w)(?:fem|sing|pl|plural)\.?(?!\w)\s*",re.I)
_UNC_RE=re.compile(r"\(\?\)"); _CDQ=re.compile("[\u201c\u201d]"); _CSQ=re.compile("[\u2018\u2019]")
_MO_RE=re.compile(r"\bMonth\s+(XII|XI|X|IX|VIII|VII|VI|V|IV|III|II|I)\b",re.I)
_R2I={"I":1,"II":2,"III":3,"IV":4,"V":5,"VI":6,"VII":7,"VIII":8,"IX":9,"X":10,"XI":11,"XII":12}
_RW_RE=re.compile(r"\b(\w+)(?:\s+\1\b)+"); _RP_RE=re.compile(r"([.,])\1+"); _PS_RE=re.compile(r"\s+([.,:])")
_FORB=str.maketrans("","","——<>⌈⌋⌊[]+ʾ;")  # () intentionally excluded — appear in ground truth
_CM_RE=re.compile(r'(?<=\s)-(gold|tax|textiles)\b')
_CR={"gold":"pašallum gold","tax":"šadduātum tax","textiles":"kutānum textiles"}
def _cm(m): return _CR[m.group(1)]
_SK=[(re.compile(r'5\s+11\s*/\s*12\s+shekels?',re.I),'6 shekels less 15 grains'),
     (re.compile(r'5\s*/\s*12\s+shekels?',re.I),'⅓ shekel 15 grains'),
     (re.compile(r'7\s*/\s*12\s+shekels?',re.I),'½ shekel 15 grains'),
     (re.compile(r'1\s*/\s*12\s*(?:\(shekel\)|\bshekel)?',re.I),'15 grains')]
_SA_RE=re.compile(r'(?<![0-9/])\s+/\s+(?![0-9])\S+')
_ST_RE=re.compile(r'<<[^>]*>>|<(?!gap\b)[^>]*>'); _ES_RE=re.compile(r'(?<!\w)(?:\.\.+|xx+)(?!\w)')
_MG_RE=re.compile(r'(?:<gap>\s*){2,}'); _HK_TR=str.maketrans({"ḫ":"h","Ḫ":"H"})
def _mon(m): return f"Month {_R2I.get(m.group(1).upper(),m.group(1))}"

class VectorizedPostprocessor:
    def postprocess_batch(self,texts):
        s=pd.Series(texts).fillna("").astype(str)
        s=_ngaps(s).str.replace(_PN_RE,"<gap>",regex=True).str.replace(_CM_RE,_cm,regex=True)
        for pat,repl in _SK: s=s.str.replace(pat,repl,regex=True)
        s=s.str.replace(_EF_RE,_fr,regex=True)
        s=s.str.replace(_FLOAT_RE,lambda m:_canon_dec(float(m.group(1))),regex=True)
        s=s.str.replace(_SG_RE," ",regex=True).str.replace(_BG_RE," ",regex=True)
        s=s.str.replace(_UNC_RE,"",regex=True).str.replace(_ST_RE,"",regex=True)
        s=s.str.replace(_ES_RE,"",regex=True).str.replace(_SA_RE,"",regex=True)
        s=s.str.replace(_CDQ,'"',regex=True).str.replace(_CSQ,"'",regex=True)
        s=s.str.replace(_MO_RE,_mon,regex=True).str.replace(_MG_RE,"<gap>",regex=True)
        s=s.str.replace("<gap>","\x00G\x00",regex=False).str.translate(_FORB)
        s=s.str.replace("\x00G\x00"," <gap> ",regex=False).str.translate(_HK_TR)
        s=s.str.replace(_RW_RE,r"\1",regex=True)
        for n in range(4,1,-1):
            s=s.str.replace(r"\b((?:\w+\s+){"+str(n-1)+r"}\w+)(?:\s+\1\b)+",r"\1",regex=True)
        s=s.str.replace(_PS_RE,r"\1",regex=True).str.replace(_RP_RE,r"\1",regex=True)
        return s.str.replace(_WS_RE," ",regex=True).str.strip().tolist()

In [10]:
# -----------------------
#   Dataset + bucketing
# -----------------------

# PyTorch Dataset wrapper to load, preprocess, and format Akkadian text for ByT5.
class AkkadianDataset(Dataset):
    def __init__(self,df,pre,logger):
        self.ids=df["id"].tolist()
        src="transliteration" if "transliteration" in df.columns else "source"
        proc=pre.preprocess_batch(df[src].fillna("").tolist())
        self.texts=["translate Akkadian to English: "+t for t in proc]
        logger.info(f"Dataset: {len(self.ids)} samples (col='{src}')")
    def __len__(self): return len(self.ids)
    def __getitem__(self,i): return self.ids[i],self.texts[i]

# Custom batch sampler that groups sentences of similar lengths to reduce padding.
class BucketBatchSampler(Sampler):
    def __init__(self,ds,bs,nb,logger):
        self.bs=bs; lengths=[len(t.split()) for _,t in ds]
        si=sorted(range(len(lengths)),key=lambda i:lengths[i]); bz=max(1,len(si)//max(1,nb))
        self.bkts=[si[i*bz:None if i==nb-1 else (i+1)*bz] for i in range(nb)]
        for i,b in enumerate(self.bkts):
            if b: logger.info(f"  Bucket {i}: {len(b)} samples, "
                              f"len[{min(lengths[x] for x in b)},{max(lengths[x] for x in b)}]")
    def __iter__(self):
        for b in self.bkts:
            for i in range(0,len(b),self.bs): yield b[i:i+self.bs]
    def __len__(self): return sum(math.ceil(len(b)/self.bs) for b in self.bkts)

# Exact MBR Selector

The `MBRSelector` class implements a streamlined, high-performance Minimum Bayes Risk (MBR) decoding strategy. Unlike previous iterations (which relied on explicit count-weighting and hardcoded agreement bonuses), this version uses a deduplicated merged pool and a uniform average, relying on the intrinsic distribution of the pool to implicitly reward cross-model agreement.

## Theoretical Motivation

When multiple models (or adjacent beam paths) converge on the same or highly similar strings, the combined generation pool becomes densely populated with these semantic equivalents. By deduplicating the pool and calculating utility against *all* unique candidates, strings that capture the consensus of the models naturally achieve higher pairwise scores. Therefore, explicit cross-model bonuses are unnecessary; the consensus signal is already mathematically embedded in the composite utility. 

## Scoring Mechanism

The selector calculates a final score for each candidate h using the following components:

**1. Uniform Pairwise Utility:**
* Instead of count-weighting, the selector calculates a simple **uniform average** over all n-1 unique references in the deduplicated pool.
* The pairwise score is an **additive composite** of three metrics, preventing the score from collapsing to zero if a single metric fails:
  * **chrF++** (Weight: 0.55)
  * **sBLEU** (Weight: 0.25)
  * **Jaccard Similarity** (Weight: 0.20)

**2. Gaussian Length Bonus:**
* A symmetric, additive bell-curve bonus is applied based on the candidate's length relative to the pool's median length. 
* It uses the formula sigma = max(median * 0.4, 5.0) to calculate the z-score.
* This gently nudges scores (by up to 10 points) to break ties and penalizes excessively short or long generations equally, without overriding the primary quality signals.

**3. Removal of Explicit Bonuses:**
* The explicit cross-model (+0.12) and within-model (+0.015) bonuses from earlier versions are intentionally **absent**. Adding them would double-count the implicit frequency signal already captured by the deduplicated uniform average.

## Implementation Details

* **Merged Deduplication:** Candidate pools from all sources are merged and deduplicated *before* scoring. A string produced by both models appears only once, but its high similarity to other candidates guarantees a competitive utility score.
* **Metric Normalization:** The final pairwise composite is normalized by the sum of its weights (`w_chrf + w_bleu + w_jaccard`) to maintain a consistent scale.
* **Scalability:** Includes a `pool_cap` (default 32) applied *after* deduplication to maintain computational efficiency during the O(n^2) pairwise utility calculations.

In [11]:
# -----------------
#   Model Wrapper
# -----------------

class ModelWrapper:
    def __init__(self,path,cfg:Config,logger,label="Model"):
        self.cfg=cfg; self.logger=logger; self.label=label
        logger.info(f"[{label}] Loading {path} …")
        self.tok=AutoTokenizer.from_pretrained(path)
        self.model=AutoModelForSeq2SeqLM.from_pretrained(path).to(DEVICE).eval()
        if DEVICE.type=="cuda":
            try: torch.set_float32_matmul_precision("high")
            except: pass
            logger.info(f"[{label}] {sum(p.numel() for p in self.model.parameters()):,} params  "
                        f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")
        if cfg.use_better_transformer and DEVICE.type=="cuda":
            try:
                from optimum.bettertransformer import BetterTransformer
                self.model=BetterTransformer.transform(self.model)
                logger.info(f"[{label}] BetterTransformer")
            except Exception as e: logger.warning(f"[{label}] BT skipped: {e}")

    def collate(self,batch):
        ids=[b[0] for b in batch]; texts=[b[1] for b in batch]
        enc=self.tok(texts,max_length=self.cfg.max_input_length,
                     padding=True,truncation=True,return_tensors="pt")
        return ids,enc

    def _eps(self,ids,attn,eps,n):
        try:
            out=self.model.generate(input_ids=ids,attention_mask=attn,do_sample=True,
                num_beams=1,epsilon_cutoff=eps,num_return_sequences=n,
                max_new_tokens=self.cfg.max_new_tokens,
                repetition_penalty=self.cfg.repetition_penalty,use_cache=True)
            return self.tok.batch_decode(out,skip_special_tokens=True)
        except Exception as e:
            self.logger.warning(f"[{self.label}] ε={eps} failed: {e}")
            return [""]*ids.shape[0]*n

    def generate_candidates(self,ids,attn,beam_size)->List[List[str]]:
        cfg=self.cfg; B=ids.shape[0]; ctx=_bf16_ctx(DEVICE,cfg.use_bf16_amp)
        Rb=cfg.num_beam_cands; Rs=cfg.num_sample_per_temp
        with ctx:
            bout=self.model.generate(input_ids=ids,attention_mask=attn,do_sample=False,
                num_beams=max(beam_size,Rb),num_return_sequences=Rb,
                max_new_tokens=cfg.max_new_tokens,length_penalty=cfg.length_penalty,
                early_stopping=cfg.early_stopping,
                repetition_penalty=cfg.repetition_penalty,use_cache=True)
            btexts=self.tok.batch_decode(bout,skip_special_tokens=True)
            ttexts=[]; nt=0
            if cfg.use_sampling:
                nt=len(cfg.sample_temperatures)
                for temp in cfg.sample_temperatures:
                    try:
                        out=self.model.generate(input_ids=ids,attention_mask=attn,
                            do_sample=True,num_beams=1,top_p=cfg.top_p,temperature=temp,
                            num_return_sequences=Rs,max_new_tokens=cfg.max_new_tokens,
                            repetition_penalty=cfg.repetition_penalty,use_cache=True)
                        ttexts.extend(self.tok.batch_decode(out,skip_special_tokens=True))
                    except Exception as e:
                        self.logger.warning(f"[{self.label}] T={temp} failed: {e}")
                        ttexts.extend([""]*B*Rs)
            Re=cfg.num_eps_cands
            etexts=self._eps(ids,attn,cfg.epsilon,Re) if Re>0 else []
        pools=[]
        for i in range(B):
            p=list(btexts[i*Rb:(i+1)*Rb])
            for t in range(nt): p.extend(ttexts[t*B*Rs+i*Rs:t*B*Rs+i*Rs+Rs])
            if etexts: p.extend(etexts[i*Re:(i+1)*Re])
            pools.append(p)
        if pools: self.logger.info(
            f"[{self.label}] hyps={len(pools[0])} (b={Rb} nuc={nt*Rs} ε={Re})")
        return pools

    def unload(self):
        try:
            from optimum.bettertransformer import BetterTransformer
            self.model=BetterTransformer.reverse(self.model)
        except: pass
        del self.model,self.tok; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache(); torch.cuda.synchronize()
            free=(torch.cuda.get_device_properties(0).total_memory
                  -torch.cuda.memory_allocated())/1e9
            self.logger.info(f"[{self.label}] Unloaded. GPU free: {free:.2f} GB")

In [12]:
# ------------------------------------
#   MBR: Hybrid's exact MBR selector
# ------------------------------------

class MBRSelector:
    def __init__(self, pool_cap: int = 32,
                 w_chrf: float = 0.55, w_bleu: float = 0.25,
                 w_jaccard: float = 0.20, w_length: float = 0.10):
        self.pool_cap  = pool_cap
        self.w_chrf    = w_chrf
        self.w_bleu    = w_bleu
        self.w_jaccard = w_jaccard
        self.w_length  = w_length
        self._pw_norm  = max(w_chrf + w_bleu + w_jaccard, 1e-9)
        if SACREBLEU_OK:
            self._chrf_m = _sacrebleu.metrics.CHRF(word_order=2)
            self._bleu_m = _sacrebleu.metrics.BLEU(effective_order=True)

    def _chrfpp(self, a: str, b: str) -> float:
        if not a or not b: return 0.0
        if SACREBLEU_OK:
            return float(self._chrf_m.sentence_score(a, [b]).score)
        return _py_chrf(a, b)

    def _bleu(self, a: str, b: str) -> float:
        if not a or not b: return 0.0
        if SACREBLEU_OK:
            try: return float(self._bleu_m.sentence_score(a, [b]).score)
            except: pass
        return _py_sbleu(a, b)

    @staticmethod
    def _jaccard(a: str, b: str) -> float:
        ta, tb = set(a.lower().split()), set(b.lower().split())
        if not ta and not tb: return 100.0
        if not ta or not tb: return 0.0
        return 100.0 * len(ta & tb) / len(ta | tb)

    def _pairwise_score(self, a: str, b: str) -> float:
        return (self.w_chrf    * self._chrfpp(a, b)
              + self.w_bleu    * self._bleu(a, b)
              + self.w_jaccard * self._jaccard(a, b)) / self._pw_norm

    @staticmethod
    def _length_bonus(lengths: List[int], idx: int) -> float:
        if not lengths: return 100.0
        median = float(np.median(lengths))
        sigma  = max(median * 0.4, 5.0)
        z      = (lengths[idx] - median) / sigma
        return 100.0 * math.exp(-0.5 * z * z)

    @staticmethod
    def _dedup(xs: List[str]) -> List[str]:
        seen, out = set(), []
        for x in xs:
            x = str(x).strip()
            if x and x not in seen:
                out.append(x); seen.add(x)
        return out

    def pick(self, candidates: List[str]) -> str:
        cands = self._dedup(candidates)
        if self.pool_cap:
            cands = cands[:self.pool_cap]

        n = len(cands)
        if n == 0: return ""
        if n == 1: return cands[0]

        lengths = [len(c.split()) for c in cands]
        scores  = []

        for i in range(n):
            # Simple uniform average — correct for a deduplicated pool
            pw = sum(
                self._pairwise_score(cands[i], cands[j])
                for j in range(n) if j != i
            ) / max(1, n - 1)

            lb    = self._length_bonus(lengths, i)
            total = pw + self.w_length * lb
            scores.append(total)

        return cands[int(np.argmax(scores))]

# Inference Engine & Orchestration

The `InferenceEngine` class is the central coordinator of the translation pipeline. It manages the end-to-end flow from raw data ingestion to the final generation of the ensemble translation, ensuring hardware resources are used efficiently and the complex decoding logic is applied correctly.

## Core Responsibilities

The engine orchestrates several critical stages of the inference process:

### 1. Data Loading & Performance Tuning
* **Bucket-Batching Orchestration (`_dl`):** Coordinates with the `BucketBatchSampler` to group inputs by length. This minimizes padding within batches, drastically increasing throughput.
* **Adaptive Beam Sizing (`_adaptive_beams`):** Dynamically adjusts the beam search width based on input length (attention mask density). Shorter sequences use fewer beams to save computation time without sacrificing quality.

### 2. Sequential Model Execution (`_run_model`)
* **Memory-Safe Execution:** Runs inference in `torch.inference_mode()` for speed and reduced VRAM usage.
* **Candidate Pooling:** Iteratively processes batches through the `ModelWrapper` to collect a comprehensive pool of translation candidates for every sentence.
* **Error Resilience:** Includes robust error handling to catch Out-of-Memory (OOM) exceptions. If a batch fails, it clears the GPU cache and proceeds with empty pools to ensure the entire dataset continues processing.

### 3. The Ensemble Pipeline (`run`)
The main `run` method executes a streamlined, memory-efficient multi-phase workflow:
* **Phase 1: Model A Inference:** Loads the primary model, generates raw candidates, then **unloads it completely** from the GPU to free VRAM.
* **Phase 2: Model B Inference:** (If in two-model mode) Loads the ensemble partner, generates raw candidates, and unloads it.
* **Phase 3: Pool Merge & Postprocessing:** Unlike previous versions, the raw candidate pools from both models are **merged first**, and then the `VectorizedPostprocessor` is applied to the combined list. This prepares a single, standardized pool of English strings.
* **Phase 4: MBR Selection:** Invokes the `MBRSelector` on the unified pool. Because the models' outputs are merged and deduplicated, the cross-model consensus signal is captured implicitly mathematically, removing the need for explicit identity tracking or hardcoded bonuses. Includes a fallback mechanism for completely damaged texts.

## Monitoring & Checkpointing

* **Periodic Checkpointing:** Automatically saves the current translation state to a CSV file at set intervals (e.g., every `checkpoint_freq` rows). This prevents total data loss in the event of a Kaggle kernel timeout or crash.
* **Final Analysis:** Logs descriptive statistics on the final output, including the total number of processed rows, average translation length, the count of empty/failed translations, and samples of the final selected outputs.

In [13]:
# --------------------
#   Inference engine
# --------------------

class InferenceEngine:
    def __init__(self, cfg: Config, logger):
        self.cfg=cfg; self.logger=logger
        self.pre=OptimizedPreprocessor(); self.post=VectorizedPostprocessor()
        self.mbr=MBRSelector(
            pool_cap=cfg.mbr_pool_cap,
            w_chrf=cfg.mbr_w_chrf,
            w_bleu=cfg.mbr_w_bleu,
            w_jaccard=cfg.mbr_w_jaccard,
            w_length=cfg.mbr_w_length)

    def _adaptive_beams(self,attn):
        if not self.cfg.use_adaptive_beams: return self.cfg.num_beams
        med=float(attn.sum(dim=1).float().median().item())
        return max(self.cfg.num_beam_cands,self.cfg.num_beams//2) if med<100 \
               else self.cfg.num_beams

    def _dl(self,ds,wrapper):
        if self.cfg.use_bucket_batching:
            return DataLoader(ds,
                batch_sampler=BucketBatchSampler(ds,self.cfg.batch_size,
                                                 self.cfg.num_buckets,self.logger),
                num_workers=self.cfg.num_workers,collate_fn=wrapper.collate,
                pin_memory=(DEVICE.type=="cuda"))
        return DataLoader(ds,batch_size=self.cfg.batch_size,shuffle=False,
                          num_workers=self.cfg.num_workers,collate_fn=wrapper.collate,
                          pin_memory=(DEVICE.type=="cuda"))

    def _run_model(self,wrapper,ds) -> Dict[str,List[str]]:
        pools={}
        with torch.inference_mode():
            for ids,enc in tqdm(self._dl(ds,wrapper),desc=f"  {wrapper.label}"):
                idt=enc.input_ids.to(DEVICE,non_blocking=True)
                att=enc.attention_mask.to(DEVICE,non_blocking=True)
                try:
                    for sid,pool in zip(ids,wrapper.generate_candidates(
                            idt,att,self._adaptive_beams(att))):
                        pools[str(sid)]=pool
                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        self.logger.error("OOM — skipping batch")
                        torch.cuda.empty_cache()
                        for sid in ids: pools.setdefault(str(sid),[])
                    else: raise
                except Exception as e:
                    self.logger.error(f"Batch error: {e}")
                    for sid in ids: pools.setdefault(str(sid),[])
                if torch.cuda.is_available(): torch.cuda.empty_cache()
        return pools

    def run(self, test_df: pd.DataFrame) -> pd.DataFrame:
        cfg=self.cfg; log=self.logger; n_mod=2 if TWO_MODEL_MODE else 1
        log.info("="*65)
        log.info(f"v41 | {'2-Model' if TWO_MODEL_MODE else '1-Model'} | "
                 f"Hybrid MBR (merged pool, uniform avg) + epsilon")
        log.info(f"  MODEL_A: {Path(cfg.model_a_path).relative_to(INPUT_ROOT)}")
        if TWO_MODEL_MODE:
            log.info(f"  MODEL_B: {Path(cfg.model_b_path).relative_to(INPUT_ROOT)}")
        log.info(f"  beam={cfg.num_beam_cands}×{cfg.num_beams} LP={cfg.length_penalty}")
        log.info(f"  nucleus temps={cfg.sample_temperatures} ×{cfg.num_sample_per_temp}"
                 f" = {cfg.num_sample_cands}")
        log.info(f"  ε=0.02: {cfg.num_eps_cands}  pool/model={cfg.cands_per_model}"
                 f"  total≈{cfg.cands_per_model*n_mod}")
        log.info(f"  MBR: chrF++×{cfg.mbr_w_chrf} + BLEU×{cfg.mbr_w_bleu}"
                 f" + Jaccard×{cfg.mbr_w_jaccard} + Gaussian×{cfg.mbr_w_length}")
        log.info(f"  MBR: MERGED pool, uniform average, NO explicit cross-model bonus")
        log.info(f"  pool_cap={cfg.mbr_pool_cap}")
        log.info("="*65)

        ds=AkkadianDataset(test_df,self.pre,log); sids=[str(s) for s in ds.ids]

        log.info(f"Phase 1/{n_mod} — Model A")
        wa=ModelWrapper(cfg.model_a_path,cfg,log,"Model-A")
        pa=self._run_model(wa,ds); wa.unload(); del wa

        pb: Dict[str,List[str]] = {}
        if TWO_MODEL_MODE:
            log.info("Phase 2/2 — Model B")
            wb=ModelWrapper(cfg.model_b_path,cfg,log,"Model-B")
            pb=self._run_model(wb,ds); wb.unload(); del wb

        log.info("Phase 3 — Pool merge + postprocess + MBR …")
        rows=[]
        for sid in tqdm(sids,desc="  MBR"):
            # Merge both model pools FIRST (as the hybrid 35.7 notebook does),
            # then postprocess the combined list together.
            # This is the key structural difference from v39/v40, where pools
            # were postprocessed separately to track model identity for the
            # cross-model bonus. Here, the cross-model signal is implicit:
            # a string produced by both models appears in the merged list twice
            # and after dedup contributes pairwise utility against all others.
            combined_raw = pa.get(sid, []) + pb.get(sid, [])
            pp = self.post.postprocess_batch(combined_raw) if combined_raw else []
            chosen = self.mbr.pick(pp)

            if not chosen or not chosen.strip():
                nz = [c for c in pp if c.strip()]
                chosen = max(nz, key=len) if nz else "The tablet is too damaged to translate."
            rows.append((sid, chosen))

            if len(rows) % cfg.checkpoint_freq == 0:
                ckpt = Path(cfg.output_dir)/f"v41_ckpt_{len(rows)}.csv"
                pd.DataFrame(rows,columns=["id","translation"]).to_csv(ckpt,index=False)
                log.info(f"  Checkpoint {len(rows)} → {ckpt}")

        df = pd.DataFrame(rows, columns=["id","translation"])
        log.info(f"Done. rows={len(df)} "
                 f"empty={df['translation'].str.strip().eq('').sum()} "
                 f"avglen={df['translation'].str.len().mean():.1f}")
        for i in range(min(4,len(df))):
            log.info(f"  [{df.iloc[i]['id']}] {str(df.iloc[i]['translation'])[:90]}")
        return df

In [14]:
# ----------
#    Main
# ----------

if __name__ == "__main__":
    cfg=Config(); logger=setup_logging(cfg.output_dir,"v41")
    print(f"\nv41 — Hybrid MBR (merged pool, uniform average, no cross-model bonus)")
    print(f"  MBR: chrF++×{cfg.mbr_w_chrf} + BLEU×{cfg.mbr_w_bleu}"
          f" + Jaccard×{cfg.mbr_w_jaccard} + Gaussian×{cfg.mbr_w_length}")
    print(f"  Pools: MERGED before postprocessing (hybrid approach)")
    print(f"  Pool: {cfg.num_beam_cands}b + {cfg.num_sample_cands}nuc"
          f" + {cfg.num_eps_cands}ε = {cfg.cands_per_model}/model"
          f" × {1+int(TWO_MODEL_MODE)} = ~{cfg.cands_per_model*(1+int(TWO_MODEL_MODE))} total")
    print(f"  Temps: {cfg.sample_temperatures} (hybrid's set)")
    print()

    test_df=pd.read_csv(cfg.test_data_path,encoding="utf-8")
    logger.info(f"Test: {len(test_df)} rows  cols={list(test_df.columns)}")
    if "id" not in test_df.columns:
        test_df=test_df.rename(columns={
            next((c for c in test_df.columns if "id" in c.lower()),test_df.columns[0]):"id"})
    if "transliteration" not in test_df.columns and "source" not in test_df.columns:
        nid=[c for c in test_df.columns if c!="id"]
        if nid: test_df=test_df.rename(columns={nid[0]:"transliteration"})

    result=InferenceEngine(cfg,logger).run(test_df)
    out=Path(cfg.output_dir)/"submission.csv"
    result.to_csv(out,index=False)
    logger.info(f"✅ {out} ({len(result)} rows)")

    with open(Path(cfg.output_dir)/"config_v41.json","w") as f:
        json.dump({"version":"v41",
            "mbr":"hybrid_merged_pool_uniform_average",
            "pairwise_weights":{
                "chrf":cfg.mbr_w_chrf,"bleu":cfg.mbr_w_bleu,"jaccard":cfg.mbr_w_jaccard},
            "length_bonus_weight":cfg.mbr_w_length,
            "cross_model_bonus":"none_implicit_via_merged_dedup",
            "pool_cap":cfg.mbr_pool_cap,
            "pool_merge":"before_postprocessing",
            "models":1+int(TWO_MODEL_MODE),
            "beam":{"cands":cfg.num_beam_cands,"beams":cfg.num_beams,
                    "LP":cfg.length_penalty,"RP":cfg.repetition_penalty},
            "nucleus_temps":cfg.sample_temperatures,
            "epsilon_02":cfg.num_eps_cands,
            "pool_per_model":cfg.cands_per_model},f,indent=2)

    print(f"\n✅ submission.csv ({len(result)} rows)")

2026-03-24 08:56:12,089 | INFO | Test: 4 rows  cols=['id', 'text_id', 'line_start', 'line_end', 'transliteration']
2026-03-24 08:56:12,099 | INFO | =================================================================
2026-03-24 08:56:12,100 | INFO | v41 | 2-Model | Hybrid MBR (merged pool, uniform avg) + epsilon
2026-03-24 08:56:12,100 | INFO |   MODEL_A: datasets/assiaben/final-byt5/byt5-akkadian-optimized-34x
2026-03-24 08:56:12,101 | INFO |   MODEL_B: datasets/spencevanasperdt/mattiaangelibyt5-akkadian-mbrpytorchdefault6
2026-03-24 08:56:12,102 | INFO |   beam=4×8 LP=1.3
2026-03-24 08:56:12,102 | INFO |   nucleus temps=[0.55, 0.75, 0.95] ×2 = 6
2026-03-24 08:56:12,103 | INFO |   ε=0.02: 6  pool/model=16  total≈32
2026-03-24 08:56:12,103 | INFO |   MBR: chrF++×0.55 + BLEU×0.25 + Jaccard×0.2 + Gaussian×0.1
2026-03-24 08:56:12,104 | INFO |   MBR: MERGED pool, uniform average, NO explicit cross-model bonus
2026-03-24 08:56:12,105 | INFO |   pool_cap=32
2026-03-24 08:56:12,105 | INFO | ====


v41 — Hybrid MBR (merged pool, uniform average, no cross-model bonus)
  MBR: chrF++×0.55 + BLEU×0.25 + Jaccard×0.2 + Gaussian×0.1
  Pools: MERGED before postprocessing (hybrid approach)
  Pool: 4b + 6nuc + 6ε = 16/model × 2 = ~32 total
  Temps: [0.55, 0.75, 0.95] (hybrid's set)



Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
2026-03-24 08:56:36,255 | INFO | [Model-A] 581,653,248 params  GPU: 2.38 GB
2026-03-24 08:56:36,256 | WARNING | [Model-A] BT skipped: No module named 'optimum'
2026-03-24 08:56:36,257 | INFO |   Bucket 0: 1 samples, len[20,20]
2026-03-24 08:56:36,258 | INFO |   Bucket 1: 1 samples, len[20,20]
2026-03-24 08:56:36,259 | INFO |   Bucket 2: 1 samples, len[23,23]
2026-03-24 08:56:36,259 | INFO |   Bucket 3: 1 samples, len[38,38]


  Model-A:   0%|          | 0/4 [00:00<?, ?it/s]

...

  Model-B:   0%|          | 0/4 [00:00<?, ?it/s]

2026-03-24 08:58:26,608 | INFO | [Model-B] hyps=16 (b=4 nuc=6 ε=6)
2026-03-24 08:58:40,499 | INFO | [Model-B] hyps=16 (b=4 nuc=6 ε=6)
2026-03-24 08:58:57,309 | INFO | [Model-B] hyps=16 (b=4 nuc=6 ε=6)
2026-03-24 08:59:25,477 | INFO | [Model-B] hyps=16 (b=4 nuc=6 ε=6)
2026-03-24 08:59:36,031 | INFO | [Model-B] Unloaded. GPU free: 15.63 GB
2026-03-24 08:59:36,032 | INFO | Phase 3 — Pool merge + postprocess + MBR …


  MBR:   0%|          | 0/4 [00:00<?, ?it/s]

2026-03-24 08:59:39,002 | INFO | Done. rows=4 empty=0 avglen=170.5
2026-03-24 08:59:39,004 | INFO |   [0] Thus kārum Kanesh, say to the <gap> dātum, our messenger, every single colony and the trad
2026-03-24 08:59:39,004 | INFO |   [1] As for the tablet of the City, whoever buys meteoric iron, it is not indebted to the proce
2026-03-24 08:59:39,005 | INFO |   [2] As soon as you have heard our letter, whether he has sold anything there for anything, or 
2026-03-24 08:59:39,006 | INFO |   [3] I sent our tablet to every single colony and the trading stations. Whether he sold meteori
2026-03-24 08:59:39,017 | INFO | ✅ /kaggle/working/submission.csv (4 rows)



✅ submission.csv (4 rows)
